# Preprocessing pipeline — Daphnet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rokhaya8/fall-detection-lstm/blob/main/notebooks/02_preprocessing_daphnet.ipynb)

Part of [fall-detection-lstm](https://github.com/Rokhaya8/fall-detection-lstm). Paths are set by `BASE_DIR` in the first code cell — adapt it to your Google Drive.

## Cellule 1 — Configuration et chargement de l'environnement
## Cellule 1 — Configuration et chargement de l'environnement

Cette cellule initialise l'environnement de travail :
- Monte Google Drive et extrait le dataset Daphnet depuis le zip
- Définit les chemins vers les données brutes et le dossier de sauvegarde
- Configure les paramètres du pipeline :

| Paramètre | Valeur | Justification |
|---|---|---|
| Window size | 200 samples | = 1 seconde à 200 Hz après rééchantillonnage — cohérent avec SisFall |
| Step size | 100 samples | Chevauchement 50% |
| Capteurs sélectionnés | Cheville (1-3) + Tronc (7-9) | 6 colonnes pour cohérence avec SisFall, cuisse écartée |

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as scipy_signal
import zipfile
from google.colab import drive

drive.mount('/content/drive')

# ── Dossier du projet dans Google Drive (à adapter si besoin) ──
import os
BASE_DIR = '/content/drive/MyDrive/Projet_Parkinson'
os.makedirs(f'{BASE_DIR}/Figures', exist_ok=True)


# Zip déjà sur ton Drive
DAPHNET_ZIP = f'{BASE_DIR}/Datasets_bruts/Daphnet_dataset.zip'

# Vérification
print(os.path.exists(DAPHNET_ZIP))

# Extraction
DAPHNET_EXTRACT_DIR = "/content/data/daphnet"
os.makedirs(DAPHNET_EXTRACT_DIR, exist_ok=True)

if len(os.listdir(DAPHNET_EXTRACT_DIR)) == 0:
    print("Extraction en cours...")
    with zipfile.ZipFile(DAPHNET_ZIP, 'r') as z:
        z.extractall(DAPHNET_EXTRACT_DIR)
    print("✓ Extraction terminée !")
else:
    print("✓ Déjà extrait")

# Configuration
# Recherche récursive des 17 fichiers SxxRyy.txt : fonctionne quelle que soit
# la structure interne du zip (site UCI, copie Kaggle...)
import re
DAPHNET_FILES = {}
for racine, _, fichiers in os.walk(DAPHNET_EXTRACT_DIR):
    for fichier in fichiers:
        if re.fullmatch(r'S\d{2}R\d{2}\.txt', fichier):
            DAPHNET_FILES.setdefault(fichier, os.path.join(racine, fichier))  # doublons ignorés
assert DAPHNET_FILES, f'Aucun fichier Daphnet trouvé dans {DAPHNET_EXTRACT_DIR}'
DAPHNET_PROCESSED_DIR = f'{BASE_DIR}/Data_Preprocessed/Daphnet'

# Processing parameters
WINDOW_SIZE = 200      # Samples per window = 1 seconde à 200 Hz après rééchantillonnage
STEP_SIZE = 100        # Stride between windows (50% overlap)
SELECTED_CHANNELS = [1, 2, 3, 7, 8, 9]  # Ankle (1-3) + Trunk (7-9)
NUM_CHANNELS = len(SELECTED_CHANNELS)

os.makedirs(DAPHNET_PROCESSED_DIR, exist_ok=True)

print(f"Fichiers Daphnet trouvés : {len(DAPHNET_FILES)}")
print(f"Output Directory: {DAPHNET_PROCESSED_DIR}")
print(f"Window size: {WINDOW_SIZE} samples (= 1s à 200Hz)")
print(f"Step size: {STEP_SIZE} samples")
print(f"Selected channels: {SELECTED_CHANNELS}")

Mounted at /content/drive
True
Extraction en cours...
✓ Extraction terminée !
TXT Directory: /content/data/daphnet/uci-daphnet-freezing-of-gait/original/dataset_fog_release/dataset
Output Directory: /content/drive/MyDrive/Projet_Parkinson/Rokhaya/Daphnet_preprocessed_200Hz
Window size: 200 samples (= 1s à 200Hz)
Step size: 100 samples
Selected channels: [1, 2, 3, 7, 8, 9]


## Cellule 2 — Chargement, filtrage et concaténation des fichiers Daphnet

Cette cellule charge et filtre les 17 fichiers Daphnet. Pour chaque fichier :

- **Sélection des 6 colonnes** : cheville (1-3) et tronc (7-9), cuisse écartée
- **Suppression des annotations 0** : échantillons hors protocole exclus
- **Mapping des labels** : annotation 1 (marche normale) → 0, annotation 2 (FOG) → 1

Les 17 fichiers sont ensuite **concaténés** en un seul tableau de shape
**(1 140 835, 6)** avec un ratio FOG de **9.7%**.

In [ ]:
# ===== ÉTAPES 2-3-4 : Chargement, filtrage et concaténation =====
txt_files = sorted(DAPHNET_FILES)
print(f"Found {len(txt_files)} TXT files")

all_data = []
all_labels = []

print("\nChargement et filtrage...")
print("-" * 60)

for i, txt_file in enumerate(txt_files, 1):
    filepath = DAPHNET_FILES[txt_file]
    data = np.loadtxt(filepath)

    # Sélection des 6 colonnes (cheville + tronc)
    X_selected = data[:, SELECTED_CHANNELS]
    y_raw = data[:, 10].astype(int)

    # Suppression annotation 0
    mask = y_raw != 0
    X_filtered = X_selected[mask]
    y_filtered = y_raw[mask]

    # Mapping labels : 1→0, 2→1
    y_mapped = y_filtered - 1

    all_data.append(X_filtered)
    all_labels.append(y_mapped)

    print(f"{i:2d}. {txt_file:15s} → {X_filtered.shape[0]:7d} échantillons | FOG: {y_mapped.mean()*100:.1f}%")

# Concaténation
X_all = np.concatenate(all_data, axis=0)
y_all = np.concatenate(all_labels, axis=0)

print("-" * 60)
print(f"\nShape totale : {X_all.shape}")
print(f"Ratio FOG    : {y_all.mean()*100:.1f}%")
print(f"✓ Chargement terminé !")

Found 17 TXT files

Chargement et filtrage...
------------------------------------------------------------
 1. S01R01.txt      →   92802 échantillons | FOG: 5.5%
 2. S01R02.txt      →   28801 échantillons | FOG: 5.4%
 3. S02R01.txt      →   25601 échantillons | FOG: 13.8%
 4. S02R02.txt      →   64961 échantillons | FOG: 12.4%
 5. S03R01.txt      →   90882 échantillons | FOG: 17.6%
 6. S03R02.txt      →   16641 échantillons | FOG: 13.9%
 7. S03R03.txt      →   21121 échantillons | FOG: 0.0%
 8. S04R01.txt      →  132482 échantillons | FOG: 0.0%
 9. S05R01.txt      →   67844 échantillons | FOG: 25.6%
10. S05R02.txt      →   65922 échantillons | FOG: 19.7%
11. S06R01.txt      →  107523 échantillons | FOG: 7.8%
12. S06R02.txt      →   19842 échantillons | FOG: 0.0%
13. S07R01.txt      →   74241 échantillons | FOG: 5.2%
14. S07R02.txt      →   28801 échantillons | FOG: 4.6%
15. S08R01.txt      →   49284 échantillons | FOG: 26.1%
16. S09R01.txt      →  111365 échantillons | FOG: 15.5%
17. S

## Cellule 3
Daphnet est enregistré à **64 Hz** tandis que SisFall est à **200 Hz**.
Sans rééchantillonnage, une fenêtre de 200 samples représenterait
**3.1 secondes** sur Daphnet contre **1 seconde** sur SisFall — les modèles
ne verraient pas la même durée temporelle.

Le rééchantillonnage est effectué via `scipy.signal.resample` **colonne par
colonne**, portant les données de **1 140 835 → 3 565 109 échantillons**
(facteur ×3.125).

**Pour les labels** : interpolation par plus proche voisin (`np.interp` +
`np.round`) pour conserver leur nature binaire (0 ou 1) après le changement
de fréquence.

**Vérification** : le ratio FOG doit rester à **9.7%** après rééchantillonnage
— confirme que les labels sont bien préservés.

In [ ]:
from scipy import signal as scipy_signal

def upsample_to_200hz(X, y, fs_original=64, fs_target=200):
    nb_original = len(X)
    nb_target = int(nb_original * fs_target / fs_original)

    print(f"Rééchantillonnage : {nb_original:,} → {nb_target:,} échantillons")
    print(f"({fs_original} Hz → {fs_target} Hz, facteur ×{fs_target/fs_original:.3f})")

    # Upsample signaux colonne par colonne
    X_up = np.zeros((nb_target, X.shape[1]), dtype=np.float32)
    for i in range(X.shape[1]):
        print(f"  Canal {i+1}/6...", end='\r')
        X_up[:, i] = scipy_signal.resample(X[:, i], nb_target)

    # Upsample labels
    indices_orig = np.linspace(0, nb_original - 1, nb_original)
    indices_target = np.linspace(0, nb_original - 1, nb_target)
    y_up = np.round(np.interp(indices_target, indices_orig, y)).astype(int)

    print(f"\n✓ Rééchantillonnage terminé !")
    print(f"Shape après : {X_up.shape}")
    print(f"Ratio FOG après : {y_up.mean()*100:.1f}%")

    return X_up, y_up

print("Rééchantillonnage en cours... (peut prendre quelques minutes)")
X_all_up, y_all_up = upsample_to_200hz(X_all, y_all)

Rééchantillonnage en cours... (peut prendre quelques minutes)
Rééchantillonnage : 1,140,835 → 3,565,109 échantillons
(64 Hz → 200 Hz, facteur ×3.125)

✓ Rééchantillonnage terminé !
Shape après : (3565109, 6)
Ratio FOG après : 9.7%


## Cellule 4 — Fenêtrage glissant

Les mêmes paramètres que SisFall sont appliqués pour garantir la cohérence
entre les deux datasets :

- **Taille** : 200 samples = **1 seconde** à 200 Hz (après rééchantillonnage)
- **Pas** : 100 samples = chevauchement de **50%**
- **Règle majoritaire** : une fenêtre est étiquetée FOG (1) si plus de 50%
  de ses samples sont annotés FOG, sinon marche normale (0)

Le fenêtrage produit **35 650 fenêtres** de shape (200, 6), soit un tenseur
final de shape **(35 650, 200, 6)**.

**Distribution des classes :**
- Class 0 (marche normale) : 32 186 fenêtres (90.3%)
- Class 1 (FOG)            : 3 464 fenêtres (9.7%)

In [ ]:
def create_sliding_windows(X, y, window_size, step_size):
    num_windows = (len(X) - window_size) // step_size + 1

    X_windowed = np.zeros((num_windows, window_size, X.shape[1]), dtype=np.float32)
    y_windowed = np.zeros(num_windows, dtype=np.int32)

    print(f"Création de {num_windows:,} fenêtres...")

    for i in range(num_windows):
        start_idx = i * step_size
        end_idx = start_idx + window_size
        X_windowed[i] = X[start_idx:end_idx].astype(np.float32)
        window_labels = y[start_idx:end_idx]
        y_windowed[i] = 0 if np.sum(window_labels) < window_size / 2 else 1

    return X_windowed, y_windowed

print("Fenêtrage glissant...")
X_windowed, y_windowed = create_sliding_windows(X_all_up, y_all_up, WINDOW_SIZE, STEP_SIZE)

print(f"\nRésultat :")
print(f"  X_windowed : {X_windowed.shape}")
print(f"  y_windowed : {y_windowed.shape}")
print(f"  Ratio FOG  : {y_windowed.mean()*100:.1f}%")
print(f"  Class 0 (marche normale) : {(y_windowed==0).sum():,}")
print(f"  Class 1 (FOG)            : {(y_windowed==1).sum():,}")

Fenêtrage glissant...
Création de 35,650 fenêtres...

Résultat :
  X_windowed : (35650, 200, 6)
  y_windowed : (35650,)
  Ratio FOG  : 9.7%
  Class 0 (marche normale) : 32,186
  Class 1 (FOG)            : 3,464


## Cellule 5 — Standardisation indépendante

Un **StandardScaler indépendant** est ajusté directement sur les données
Daphnet — le scaler SisFall n'est pas réutilisé.

**Pourquoi un scaler indépendant ?**
L'application du scaler SisFall sur Daphnet a été testée et a produit des
valeurs aberrantes (moyenne = 380.77, std = 697.20), confirmant que les
deux datasets ont des distributions trop différentes pour partager le même
scaler. Cela s'explique par :
- Des unités différentes (SisFall converti de bits en G via formule capteur,
  Daphnet en mg divisé par 1000)
- Des amplitudes différentes (chutes jusqu'à ±16g vs marche pathologique ±2g)
- Des placements de capteurs différents (taille vs cheville et tronc)

Le StandardScaler indépendant ramène correctement les données à
**moyenne ≈ 0** et **std ≈ 1** par canal.

Le scaler est sauvegardé (`scaler_daphnet`) pour une éventuelle
réutilisation future.

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

def normalize_data(X):
    nb, taille, features = X.shape

    # Reshape 3D → 2D
    X_2d = X.reshape(-1, features)

    # Fit et transform
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_2d)

    # Reshape 2D → 3D
    X_scaled = X_scaled.reshape(nb, taille, features)

    print(f"Moyenne après normalisation : {X_scaled.mean():.6f} (≈ 0)")
    print(f"Std après normalisation     : {X_scaled.std():.6f} (≈ 1)")

    return X_scaled, scaler

print("Normalisation avec StandardScaler...")
X_normalized, scaler_daphnet = normalize_data(X_windowed)
print("✓ Normalisation terminée !")

Normalisation avec StandardScaler...
Moyenne après normalisation : -0.000000 (≈ 0)
Std après normalisation     : 1.000000 (≈ 1)
✓ Normalisation terminée !


## Cellule 6 — Sauvegarde des fichiers préprocessés

Sauvegarde des trois fichiers produits par le pipeline sur Google Drive :

- **X_daphnet_200Hz.npy** : tenseur de shape (35 650, 200, 6) contenant
  les signaux normalisés prêts pour l'inférence
- **y_daphnet_200Hz.npy** : vecteur de shape (35 650,) contenant
  les labels binaires correspondants (0 = marche normale, 1 = FOG)
- **scaler_daphnet.pkl** : StandardScaler ajusté sur Daphnet,
  sauvegardé pour une éventuelle réutilisation future

In [ ]:
import os
import pickle

DAPHNET_PROCESSED_DIR = f'{BASE_DIR}/Data_Preprocessed/Daphnet'
os.makedirs(DAPHNET_PROCESSED_DIR, exist_ok=True)

# Sauvegarde des tenseurs
X_save = os.path.join(DAPHNET_PROCESSED_DIR, "X_daphnet_200Hz.npy")
y_save = os.path.join(DAPHNET_PROCESSED_DIR, "y_daphnet_200Hz.npy")
scaler_save = os.path.join(DAPHNET_PROCESSED_DIR, "scaler_daphnet.pkl")

np.save(X_save, X_normalized)
np.save(y_save, y_windowed)

with open(scaler_save, 'wb') as f:
    pickle.dump(scaler_daphnet, f)

print(" Sauvegarde terminée !")
print(f"  X_daphnet_200Hz.npy : {X_normalized.shape}")
print(f"  y_daphnet_200Hz.npy : {y_windowed.shape}")
print(f"  scaler_daphnet.pkl  : StandardScaler sauvegardé")
print(f"  Dossier : {DAPHNET_PROCESSED_DIR}")

✅ Sauvegarde terminée !
  X_daphnet_200Hz.npy : (35650, 200, 6)
  y_daphnet_200Hz.npy : (35650,)
  scaler_daphnet.pkl  : StandardScaler sauvegardé
  Dossier : /content/drive/MyDrive/Projet_Parkinson/Pipelines_Preprocessing/Rokhaya's Version
